# Notebook 10 — Schema and Types

**Datasets:** `samples.bakehouse.sales_transactions` · `samples.bakehouse.sales_customers` · `samples.tpch.orders`

**Difficulty:** Hard

**Topics covered:**
- `printSchema`, `.dtypes`, `.schema`
- `StructType` / `StructField` — explicit schema definitions
- Schema inference vs explicit schema
- Nested struct columns with `F.struct`
- DDL string schema with `T.StructType.fromDDL`
- `ArrayType` and `MapType` columns
- Safe casting with `F.col(...).cast(...)` and `F.try_cast` / `F.when`
- Schema comparison between two DataFrames
- Schema evolution simulation (add / rename / drop columns)

> **Note:** Run cells top-to-bottom. All problems use the DataFrames loaded in the Setup cell.

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T

spark = SparkSession.builder.getOrCreate()

transactions = spark.table("samples.bakehouse.sales_transactions")
customers    = spark.table("samples.bakehouse.sales_customers")
orders       = spark.table("samples.tpch.orders")

print("transactions schema:"); transactions.printSchema()
print("customers schema:");    customers.printSchema()
print("orders schema:");       orders.printSchema()

## Problem 1 — Inspect and compare schemas

Use `.dtypes` and `.schema` to inspect `sales_transactions`. Build a result DataFrame showing `column_name`, `data_type`, and `nullable` for every column.

**Expected output columns:** `column_name`, `data_type`, `nullable`

```python
rows = [(f.name, str(f.dataType), f.nullable) for f in transactions.schema]
result_1 = spark.createDataFrame(rows, ["column_name", "data_type", "nullable"])
```

In [ ]:
# Problem 1 — write your solution here
# Assign result to: result_1
result_1 = None  # replace this


In [ ]:
# ── Tests for Problem 1 ──────────────────────────────────────────────────────
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'column_name' in cols, "Missing: column_name"
assert 'data_type' in cols, "Missing: data_type"
assert 'nullable' in cols, "Missing: nullable"
cnt = result_1.count()
assert cnt >= len(transactions.columns), "Should have at least as many rows as columns in transactions"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2 — Define an explicit StructType schema

Write a full `StructType` schema for `sales_transactions` with stricter types (`transactionID` as `LongType`, `dateTime` as `TimestampType`, `unitPrice` / `totalPrice` / `quantity` as `LongType`, others as `StringType` or `LongType` as appropriate).

Then read `sales_transactions` with `spark.table()` and demonstrate that explicit casting works by creating:
- `transactionID` cast to `LongType`
- `product` as-is (StringType)
- `totalPrice_as_double` — `totalPrice` cast to `DoubleType`

**Expected output columns:** `transactionID`, `product`, `totalPrice_as_double`



In [ ]:
# Problem 2 — write your solution here
# then select + cast the needed columns
# Assign result to: result_2
result_2 = None  # replace this


In [ ]:
# ── Tests for Problem 2 ──────────────────────────────────────────────────────
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'transactionid' in cols, "Missing: transactionID"
assert 'product' in cols, "Missing: product"
assert 'totalprice_as_double' in cols, "Missing: totalPrice_as_double"
dt = dict(result_2.dtypes)
assert dt.get('totalPrice_as_double') in ('double', 'float'), \
    f"totalPrice_as_double should be double, got {dt.get('totalPrice_as_double')}"
cnt = result_2.count()
assert cnt > 0, "Got 0 rows"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3 — Create a nested struct column

From `sales_customers`, build a DataFrame that contains:
- `customerID`
- `full_name` — concatenation of first and last name
- `address_struct` — a `StructType` column containing `city`, `state`, `country` (use `F.struct`)

Also project `address_struct.country` as a flat column to demonstrate nested field access.

**Expected output columns:** `customerID`, `full_name`, `address_struct`, `country` (from `address_struct.country`)



In [ ]:
# Problem 3 — write your solution here
# Assign result to: result_3
result_3 = None  # replace this


In [ ]:
# ── Tests for Problem 3 ──────────────────────────────────────────────────────
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'customerid' in cols, "Missing: customerID"
assert 'full_name' in cols, "Missing: full_name"
assert 'address_struct' in cols, "Missing: address_struct"
# Check that address_struct is a StructType
struct_field = result_3.schema['address_struct']
assert isinstance(struct_field.dataType, T.StructType), "address_struct should be StructType"
struct_cols = [f.name.lower() for f in struct_field.dataType.fields]
assert 'country' in struct_cols, "address_struct missing country field"
cnt = result_3.count()
assert cnt > 0, "Got 0 rows"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4 — Schema from a DDL string

Use `T.StructType.fromDDL(...)` to define a schema with these columns:

```sql
orderkey BIGINT, custkey BIGINT, status STRING, totalprice DOUBLE, orderdate DATE
```

Apply it to `tpch.orders` by selecting and casting the appropriate columns to match. Return the re-cast DataFrame.

**Expected output columns:** `orderkey`, `custkey`, `status`, `totalprice`, `orderdate`



In [ ]:
# Problem 4 — write your solution here
# Then select + cast o_orderkey as orderkey, o_custkey as custkey, etc.
# Assign result to: result_4
result_4 = None  # replace this


In [ ]:
# ── Tests for Problem 4 ──────────────────────────────────────────────────────
assert result_4 is not None, "result_4 is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
for expected in ['orderkey', 'custkey', 'status', 'totalprice', 'orderdate']:
    assert expected in cols, f"Missing: {expected}"
dt = dict(result_4.dtypes)
assert dt.get('totalprice') in ('double', 'float', 'decimal(18,2)'), \
    f"totalprice should be numeric, got {dt.get('totalprice')}"
cnt = result_4.count()
assert cnt > 0, "Got 0 rows"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5 — ArrayType and MapType columns

From `sales_transactions`, grouped by `franchiseID`, build a DataFrame containing:
- `franchiseID`
- `products_array` — `ArrayType(StringType)` — from `F.collect_list('product')`
- `payment_count_map` — `MapType(StringType, LongType)` — from `F.map_from_arrays` on aggregated payment methods and their counts

**Expected output columns:** `franchiseID`, `products_array`, `payment_count_map`



In [ ]:
# Problem 5 — write your solution here
# or two-step: groupBy franchiseID+paymentMethod, count, then groupBy franchiseID with map_from_arrays
# Assign result to: result_5
result_5 = None  # replace this


In [ ]:
# ── Tests for Problem 5 ──────────────────────────────────────────────────────
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'franchiseid' in cols, "Missing: franchiseID"
assert 'products_array' in cols, "Missing: products_array"
assert 'payment_count_map' in cols, "Missing: payment_count_map"
arr_type = result_5.schema['products_array'].dataType
assert isinstance(arr_type, T.ArrayType), "products_array should be ArrayType"
map_type = result_5.schema['payment_count_map'].dataType
assert isinstance(map_type, T.MapType), "payment_count_map should be MapType"
cnt = result_5.count()
assert cnt > 0, "Got 0 rows"
print(f"Problem 5 passed ✓  ({cnt} rows)")

## Problem 6 — Schema mismatch: safe casting

`sales_customers` has `customerID` as `bigint`. Demonstrate safe casting:

1. Cast `customerID` to `StringType` → `zip_as_string`
2. Cast `zip_as_string` back to `IntegerType` → `zip_as_int`
3. Use `F.when(F.col('zip_as_string').rlike(r'^\d+$'), F.col('zip_as_string').cast(T.IntegerType()))` to show a safe conditional cast.

**Expected output columns:** `customerID`, `zip_as_string`, `zip_as_int`



In [ ]:
# Problem 6 — write your solution here
# Assign result to: result_6
result_6 = None  # replace this


In [ ]:
# ── Tests for Problem 6 ──────────────────────────────────────────────────────
assert result_6 is not None, "result_6 is None"
assert hasattr(result_6, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'customerid' in cols, "Missing: customerID"
assert 'zip_as_string' in cols, "Missing: zip_as_string"
assert 'zip_as_int' in cols, "Missing: zip_as_int"
dt = dict(result_6.dtypes)
assert dt.get('zip_as_string') == 'string', f"zip_as_string should be string, got {dt.get('zip_as_string')}"
assert dt.get('zip_as_int') in ('int', 'integer', 'bigint', 'long'), \
    f"zip_as_int should be int/bigint, got {dt.get('zip_as_int')}"
cnt = result_6.count()
assert cnt > 0, "Got 0 rows"
print(f"Problem 6 passed ✓  ({cnt} rows)")

## Problem 7 — Compare schemas of two DataFrames

Write a **Python function** (not a UDF) that compares the schemas of two DataFrames and returns a `dict` with:
- `only_in_df1` — columns in df1 but not df2
- `only_in_df2` — columns in df2 but not df1
- `type_mismatch` — columns in both with different types

Apply it to compare `sales_transactions` vs `sales_customers`.

Convert the result to a Spark DataFrame with columns: `column_name`, `present_in`, `type_in_transactions`, `type_in_customers`.

**Expected output columns:** `column_name`, `present_in`, `type_in_transactions`, `type_in_customers`



In [ ]:
# Problem 7 — write your solution here
# Assign result to: result_7
result_7 = None  # replace this


In [ ]:
# ── Tests for Problem 7 ──────────────────────────────────────────────────────
assert result_7 is not None, "result_7 is None"
assert hasattr(result_7, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'column_name' in cols, "Missing: column_name"
assert 'present_in' in cols, "Missing: present_in"
assert 'type_in_transactions' in cols, "Missing: type_in_transactions"
assert 'type_in_customers' in cols, "Missing: type_in_customers"
cnt = result_7.count()
assert cnt > 0, "Got 0 rows — transactions and customers share no columns, so differences should exist"
print(f"Problem 7 passed ✓  ({cnt} rows)")

## Problem 8 — Schema evolution simulation

Simulate three schema evolution operations on `sales_transactions`:

1. **Add column:** `.withColumn('discount_pct', F.lit(0.0))`
2. **Rename column:** `.withColumnRenamed('paymentMethod', 'payment_method_v2')`
3. **Drop column:** `.drop('cardNumber')`

Apply all three and build a **change-log** DataFrame describing what was done.

**Expected output columns:** `change_type`, `column_name`, `detail`

Also return `result_8` as the **evolved DataFrame** (after all 3 changes). The test checks the change-log is a valid Spark DataFrame with the correct columns.



In [ ]:
# Problem 8 — write your solution here
# Step 1: apply the three schema-evolution ops to get evolved_df
# Step 2: build a change_log DataFrame
# Assign the change_log DataFrame to result_8
# Assign result to: result_8
result_8 = None  # replace this (the change_log DataFrame)


In [ ]:
# ── Tests for Problem 8 ──────────────────────────────────────────────────────
assert result_8 is not None, "result_8 is None"
assert hasattr(result_8, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_8.columns]
assert 'change_type' in cols, "Missing: change_type"
assert 'column_name' in cols, "Missing: column_name"
assert 'detail' in cols, "Missing: detail"
cnt = result_8.count()
assert cnt >= 3, f"Expected at least 3 change-log rows (ADD/RENAME/DROP), got {cnt}"
change_types = {r['change_type'] for r in result_8.select('change_type').collect()}
assert 'ADD' in change_types or 'add' in change_types, "Expected an ADD entry in change_log"
print(f"Problem 8 passed ✓  ({cnt} rows)")